In [ ]:
import pickle
import pandas as pd
import numpy as np
import os
from autotest_exp import load_experiment_result, read_config, add_labels_to_result_dfs, experiments

In [ ]:
config_file_path = './config.ini'
result_df = {}

In [ ]:
pipeline_options = [3, 1, 0]
label_multiplier = [2, 6, 12]
config = read_config(config_file_path)
executions, runs = experiments(pipeline_options, label_multiplier, config_file_path)

In [ ]:
config = read_config(config_file_path)
result_dfs, results_per_table = load_experiment_result(executions, config)
result_dfs = add_labels_to_result_dfs(result_dfs)
for execution in results_per_table.keys():
    for run in results_per_table[execution].keys():
        results_per_table[execution][run] = pd.DataFrame(results_per_table[execution][run]).T
        results_per_table[execution][run]["total_cells"] = results_per_table[execution][run]["tp"] + results_per_table[execution][run]["fp"] + results_per_table[execution][run]["fn"] + results_per_table[execution][run]["tn"]
        results_per_table[execution][run] = results_per_table[execution][run].T
runs = list(result_dfs[executions[1]].keys())

In [ ]:
comparison_dfs = {}
for run in runs:
    comparison_df = results_per_table[executions[1]][run].T
    diff_columns = ["tp", "fp", "fn", "tn", "precision", "recall", "f_score"]
    comparison_df[diff_columns] = results_per_table[executions[1]][run].T[diff_columns] - results_per_table[executions[0]][run].T[diff_columns]
    comparison_dfs[run] = comparison_df

In [ ]:
for i in [0, 1]:
    for run in runs:
        foo = result_dfs[executions[i]][run]
        foo['training_label'] = (foo['propagated_label'] == 1) | (foo['propagated_label'] == -1)
        conditions = [
            (foo["training_label"] == 1) & (foo["label"] == 1),
            (foo["training_label"] == 1) & (foo["label"] == 0),
            (foo["training_label"] == 0) & (foo["label"] == 0),
            (foo["training_label"] == 0) & (foo["label"] == 1),
        ]
        choices = ["TP", "FP", "TN", "FN"]
        foo['training_result'] =  np.select(conditions, choices, default="Unknown")

In [ ]:
e, r = (1, 2)
print(f"{executions[e]} - {runs[r]}")
foo = pd.DataFrame(result_dfs[executions[e]][runs[r]])
foo

In [ ]:
average_f_score = {}
average_recall = {}
average_precision = {}
for e in [0, 1]:
    average_f_score[executions[e]] = {}
    average_recall[executions[e]] = {}
    average_precision[executions[e]] = {}
    for r in [0, 1, 2]:
        foo = results_per_table[executions[e]][runs[r]].T
        average_recall[executions[e]][r] = pd.DataFrame(results_per_table[executions[e]][runs[r]]).T["recall"].sum() / 96
        average_precision[executions[e]][r] = pd.DataFrame(results_per_table[executions[e]][runs[r]]).T["precision"].sum() / 96
        average_f_score[executions[e]][runs[r]] = pd.DataFrame(results_per_table[executions[e]][runs[r]]).T["f_score"].sum() / 96
average_f_score = pd.DataFrame(average_f_score)
average_recall = pd.DataFrame(average_recall)
average_precision = pd.DataFrame(average_precision)
average_f_score["diff"] = average_f_score["Integration_Option_1"] - average_f_score["Integration_Option_0"]
average_recall["diff"] = average_recall["Integration_Option_1"] - average_recall["Integration_Option_0"]
average_precision["diff"] = average_precision["Integration_Option_1"] - average_precision["Integration_Option_0"]
print("average_f_score")
display(average_f_score)
print("average_recall")
display(average_recall)
print("average_precision")
display(average_precision)

In [ ]:
all_analysis = pd.DataFrame()
precision, recall, f_score= {}, {}, {}
for run in runs:
    precision[run[21:]], recall[run[21:]], f_score[run[21:]]= {}, {}, {}
    for execution in executions:
        result_df = result_dfs[execution][run]
        #result_df = result_df[result_df['table_name'] != "Rates_of_Preventable_Hospitalizations_for_Selected_Medical_Conditions_by_County_(LGHC_Indicator)"]
        analysis = result_df["result"].value_counts()
        if "TP" not in analysis: analysis["TP"] = 0
        if "FP" not in analysis: analysis["FP"] = 0
        if "TN" not in analysis: analysis["TN"] = 0
        if "FN" not in analysis: analysis["FN"] = 0

        analysis["precision"] = analysis["TP"] / (analysis["TP"] + analysis["FP"])
        analysis["recall"] = analysis["TP"] / (analysis["TP"] + analysis["FN"])
        analysis["F1 Score"] = 2 * (analysis["precision"] * analysis["recall"]) / (analysis["precision"] + analysis["recall"])
        precision[run[21:]][execution] = analysis["precision"]
        recall[run[21:]][execution] = analysis["recall"]
        f_score[run[21:]][execution] = analysis["F1 Score"]
        all_analysis[f"{run[21:]} - {execution}"] = analysis
display(all_analysis.T)
print("precision")
precision = pd.DataFrame(precision).T
precision["diff"] = precision[executions[1]] - precision[executions[0]]
display(pd.DataFrame(precision))
print("recall")
recall = pd.DataFrame(recall).T
recall["diff"] = recall[executions[1]] - recall[executions[0]]
display(pd.DataFrame(recall))
print("F1 Score")
f_score = pd.DataFrame(f_score).T
f_score["diff"] = f_score[executions[1]] - f_score[executions[0]]
display(pd.DataFrame(f_score))


In [ ]:
result_dfs["Integration_Option_0"]["_test_edbt_DGov_Typo_192_labels"]

In [ ]:


final_results = {}
folder = "output/DGov_Typo_high_TP_ratio_0/_test_edbt_DGov_Typo_subsets/DGov_Typo_high_TP_ratio_30_labels/results/final_results"
files = os.listdir(folder)
files = sorted(files)
for file in files:
    path = os.path.join(folder, file)
    with open(path, "rb") as f:
        final_results[file] = pickle.load(f)

In [ ]:
for key in final_results:
    print(key)

In [ ]:
final_results["y_labeled_by_user_all.pickle"]

In [ ]:
os.listdir("/home/micro/Documents/Code/AutoMatelda/datasets/DGov_Typo_subsets/DGov_Typo_5")

In [ ]:
with open("output/DGov_Typo_high_TP_ratio_Integration_Option_0/_test_edbt_DGov_Typo_subsets/DGov_Typo_high_TP_ratio_10_labels/results/results_df.pickle", "rb") as f:
    results_df = pickle.load(f)
results_df

In [ ]:
import pandas as pd
auto_test_output_path = "/home/pavel/AutoTest/code/AutoTest/results/detected_outliers/combined_sdc_on_305b_Assessed_2008_Lake.csv"

auto_test_output_df = pd.read_table(auto_test_output_path, dtype=str)
if auto_test_output_df.columns == ',header,outlier,conf,dist_val,SDC':
    auto_test_output_df = pd.read_csv(auto_test_output_path, dtype=str)

In [ ]:
if auto_test_output_df.columns == 'header,outlier':
    print("true")
else:
    print("false")

In [ ]:
auto_test_config = {}
auto_test_config["rerun"] = True
auto_test_config["auto_test_path"] = "/home/pavel/AutoTest/code/AutoTest"
auto_test_config["sdc_file_name"] = "combined_sdc.csv"
auto_test_config["integration_pipeline_option"] = 0

import marshmallow_pipeline.auto_test.auto_test_util as autotest

foo = autotest.load_autotest_df(auto_test_config, "output/DGov_Typo_Integration_Option_0/_test_edbt_DGov_Typo_10_labels", "BLM_AZ_Travel_Management_Plans_(Polygon).csv")

In [ ]:
foo